<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/03_AI_Engineer/intermediate/06_ai_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating AI Systems

Build robust evaluation pipelines for RAG systems and LLM applications — from RAGAS metrics to LLM-as-judge frameworks.

**Topics:** RAGAS metrics, faithfulness, answer relevance, LLM-as-judge, custom eval frameworks, golden datasets

## 1. RAGAS — RAG Assessment Framework

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from datasets import Dataset
import pandas as pd

# RAGAS evaluation dataset format
eval_dataset = Dataset.from_dict({
    'question': [
        'What is BERT used for?',
        'What training objective does BERT use?',
        'Who invented transformers?',
    ],
    'answer': [
        'BERT is used for text classification, NER, and question answering.',
        'BERT uses masked language modeling and next sentence prediction.',
        'Transformers were introduced by Vaswani et al. in 2017.',
    ],
    'contexts': [
        ['BERT (Bidirectional Encoder Representations from Transformers) achieves state-of-the-art results on 11 NLP tasks including question answering, text classification, and named entity recognition.'],
        ['BERT is pre-trained on two tasks: Masked Language Modeling (MLM) where 15% of tokens are masked, and Next Sentence Prediction (NSP).'],
        ['The transformer architecture was introduced in "Attention Is All You Need" by Vaswani et al. (2017) at Google Brain.'],
    ],
    'ground_truth': [
        'BERT is used for text classification, question answering, and NER tasks.',
        'BERT uses masked language modeling (MLM) and next sentence prediction (NSP).',
        'Transformers were invented by Vaswani et al. at Google in 2017.',
    ],
})

# Run evaluation
# results = evaluate(eval_dataset, metrics=[faithfulness, answer_relevancy, context_recall, context_precision])

# RAGAS metrics explained
metrics_explained = [
    ('Faithfulness', 'Is the answer supported by the context? (hallucination check)', 'answer + contexts → 0-1'),
    ('Answer Relevancy', 'Does the answer address the question?', 'question + answer → 0-1'),
    ('Context Recall', 'Does the context contain the needed info? (retrieval quality)', 'ground_truth + contexts → 0-1'),
    ('Context Precision', 'Are retrieved chunks relevant? (no noise)', 'question + contexts → 0-1'),
]

print('RAGAS metrics:')
print(f'{"Metric":<20} {"Measures":<50} {"Inputs"}')
print('-' * 90)
for name, desc, inputs in metrics_explained:
    print(f'{name:<20} {desc:<50} {inputs}')

# Simulated RAGAS scores
simulated_scores = pd.DataFrame({
    'question': ['What is BERT?', 'BERT training?', 'Who invented transformers?'],
    'faithfulness': [0.92, 0.88, 1.00],
    'answer_relevancy': [0.95, 0.91, 0.97],
    'context_recall': [0.89, 0.94, 0.85],
    'context_precision': [0.78, 0.82, 0.91],
})
print(f'\nSimulated RAGAS results:')
print(simulated_scores.to_string(index=False))
print(f'\nAggregate: faithfulness={simulated_scores.faithfulness.mean():.3f}')

## 2. LLM-as-Judge Evaluation Pattern

In [ ]:
import json
import os
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

class EvalResult(BaseModel):
    score: int = Field(ge=1, le=5, description='Score from 1 (worst) to 5 (best)')
    reasoning: str = Field(description='Step-by-step reasoning for the score')
    passed: bool = Field(description='Whether the response meets minimum quality bar')

JUDGE_PROMPT = """You are an expert evaluator assessing AI assistant responses.

Evaluate the response on these criteria:
1. Accuracy: Is the information factually correct?
2. Completeness: Does it fully address the question?
3. Clarity: Is it easy to understand?
4. Conciseness: Is it appropriately brief without being incomplete?
5. Helpfulness: Would a user find this genuinely useful?

Question: {question}
Response to evaluate: {response}
{reference_section}

Provide your evaluation as JSON with keys: score (1-5), reasoning (str), passed (bool).
passed=true if score >= 4."""

def llm_judge(
    question: str,
    response: str,
    reference: str = None,
    judge_model: str = 'gpt-4o',  # use strongest model for judging
) -> EvalResult:
    reference_section = f'Reference answer: {reference}' if reference else ''
    
    result = client.beta.chat.completions.parse(
        model=judge_model,
        messages=[{'role': 'user', 'content': JUDGE_PROMPT.format(
            question=question, response=response, reference_section=reference_section,
        )}],
        response_format=EvalResult,
    )
    return result.choices[0].message.parsed

# Simulated evaluation
test_cases = [
    ('What is gradient descent?', 'Gradient descent minimizes a loss function by iteratively moving in the direction of steepest descent.', 4, True),
    ('What is gradient descent?', 'It is a math thing used in machine learning.', 2, False),
]

print('LLM-as-judge results (simulated):')
for q, resp, score, passed in test_cases:
    print(f'  Q: {q}')
    print(f'  Response: {resp[:60]}...')
    print(f'  Score: {score}/5, Passed: {passed}')
    print()

## 3. Building a Custom Eval Framework

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional
import numpy as np

@dataclass
class EvalCase:
    question: str
    ground_truth: str
    context: Optional[str] = None
    tags: list[str] = field(default_factory=list)

@dataclass
class EvalMetric:
    name: str
    fn: Callable[[str, str, Optional[str]], float]  # (response, ground_truth, context) -> score
    threshold: float = 0.7

class EvalPipeline:
    """Configurable evaluation pipeline for AI systems."""
    
    def __init__(self, metrics: list[EvalMetric], system_under_test: Callable):
        self.metrics = metrics
        self.sut = system_under_test  # function: question -> response
    
    def run(self, eval_cases: list[EvalCase]) -> dict:
        results = []
        for case in eval_cases:
            response = self.sut(case.question)
            scores = {
                m.name: m.fn(response, case.ground_truth, case.context)
                for m in self.metrics
            }
            passed = all(scores[m.name] >= m.threshold for m in self.metrics)
            results.append({'case': case, 'response': response, 'scores': scores, 'passed': passed})
        
        return self._aggregate(results)
    
    def _aggregate(self, results: list) -> dict:
        agg = {}
        for m in self.metrics:
            scores = [r['scores'][m.name] for r in results]
            agg[m.name] = {'mean': np.mean(scores), 'min': np.min(scores), 'p10': np.percentile(scores, 10)}
        agg['pass_rate'] = sum(r['passed'] for r in results) / len(results)
        return agg

# Sample metrics
def exact_match(response: str, ground_truth: str, context=None) -> float:
    return float(ground_truth.lower().strip() in response.lower())

def token_f1(response: str, ground_truth: str, context=None) -> float:
    pred_tokens = set(response.lower().split())
    gt_tokens = set(ground_truth.lower().split())
    if not gt_tokens: return 1.0
    precision = len(pred_tokens & gt_tokens) / len(pred_tokens) if pred_tokens else 0
    recall = len(pred_tokens & gt_tokens) / len(gt_tokens)
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

# Demo
response = 'BERT uses masked language modeling and next sentence prediction for pre-training.'
ground_truth = 'BERT is pre-trained with masked language modeling and next sentence prediction.'
print(f'Exact match: {exact_match(response, ground_truth):.2f}')
print(f'Token F1:    {token_f1(response, ground_truth):.2f}')

## 4. Regression Testing for AI Systems

In [ ]:
import json
from pathlib import Path
from datetime import datetime

class AIRegressionTest:
    """Prevent quality regressions when updating models or prompts."""
    
    def __init__(self, baseline_path: str = './eval_baselines'):
        self.baseline_dir = Path(baseline_path)
        self.baseline_dir.mkdir(exist_ok=True)
    
    def save_baseline(self, version: str, scores: dict):
        """Save current scores as the baseline to compare against."""
        baseline = {
            'version': version,
            'timestamp': datetime.now().isoformat(),
            'scores': scores,
        }
        (self.baseline_dir / f'{version}.json').write_text(json.dumps(baseline, indent=2))
    
    def compare(self, current_scores: dict, baseline_version: str, tolerance: float = 0.02) -> dict:
        """Compare current scores against baseline. Fail if regression > tolerance."""
        baseline_path = self.baseline_dir / f'{baseline_version}.json'
        baseline = json.loads(baseline_path.read_text())['scores']
        
        regressions = []
        improvements = []
        
        for metric, current in current_scores.items():
            if isinstance(current, dict):
                current_val = current.get('mean', 0)
                baseline_val = baseline.get(metric, {}).get('mean', 0)
            else:
                current_val = current
                baseline_val = baseline.get(metric, 0)
            
            delta = current_val - baseline_val
            if delta < -tolerance:
                regressions.append(f'{metric}: {baseline_val:.3f} → {current_val:.3f} ({delta:+.3f})')
            elif delta > tolerance:
                improvements.append(f'{metric}: {baseline_val:.3f} → {current_val:.3f} ({delta:+.3f})')
        
        return {
            'passed': len(regressions) == 0,
            'regressions': regressions,
            'improvements': improvements,
        }

# Simulated regression test
baseline_scores = {'faithfulness': {'mean': 0.88}, 'answer_relevancy': {'mean': 0.91}, 'pass_rate': 0.85}
current_scores = {'faithfulness': {'mean': 0.85}, 'answer_relevancy': {'mean': 0.93}, 'pass_rate': 0.83}

print('Regression test results:')
print('Baseline vs Current:')
for metric in baseline_scores:
    b = baseline_scores[metric]
    c = current_scores[metric]
    b_val = b['mean'] if isinstance(b, dict) else b
    c_val = c['mean'] if isinstance(c, dict) else c
    delta = c_val - b_val
    status = 'REGRESSION' if delta < -0.02 else ('IMPROVEMENT' if delta > 0.02 else 'STABLE')
    print(f'  {metric:<25}: {b_val:.3f} → {c_val:.3f} ({delta:+.3f}) [{status}]')

## 5. Golden Dataset Creation and Maintenance

In [ ]:
from dataclasses import dataclass, field
from typing import Optional
import hashlib
import json

@dataclass
class GoldenExample:
    question: str
    answer: str
    context: Optional[str] = None
    difficulty: str = 'medium'  # easy, medium, hard
    tags: list[str] = field(default_factory=list)
    human_verified: bool = False
    
    @property
    def id(self) -> str:
        return hashlib.md5(self.question.encode()).hexdigest()[:8]

def generate_golden_dataset(
    documents: list[str],
    n_examples: int = 50,
    difficulty_distribution: dict = None,
) -> list[GoldenExample]:
    """Generate a golden dataset using an LLM.
    
    Steps:
    1. LLM reads each document and generates Q&A pairs
    2. Another LLM verifies the answers
    3. Human review flags remaining uncertain cases
    """
    dist = difficulty_distribution or {'easy': 0.3, 'medium': 0.5, 'hard': 0.2}
    
    GENERATE_PROMPT = """Read this document and create {n} diverse Q&A pairs.
    Include easy, medium, and hard questions.
    Format as JSON list with keys: question, answer, difficulty.
    
    Document: {doc}"""
    
    golden_examples = []
    # In practice: call LLM for each doc, parse JSON, validate, store
    return golden_examples

# Golden dataset best practices
best_practices = [
    ('Size', 'Minimum 100 examples; 500+ for production systems'),
    ('Coverage', 'Cover all topic areas, edge cases, and difficulty levels'),
    ('Verification', 'At least 30% human-verified; LLM-generated is ok for rest'),
    ('Versioning', 'Version your golden dataset with git — track changes over time'),
    ('Refresh', 'Add new examples when users report failures in production'),
    ('Balance', 'Even difficulty distribution prevents misleadingly high overall scores'),
    ('Negatives', 'Include questions with no answer in context (should say "I don\'t know")'),
]

print('Golden dataset best practices:')
for aspect, guidance in best_practices:
    print(f'  {aspect:<15}: {guidance}')

## Exercises

1. **RAGAS deep dive:** Build a RAGAS evaluation pipeline that tests your RAG system on 30 questions across 3 topic areas. Identify which topic area has the lowest faithfulness and debug why (is it a retrieval problem or a generation problem?).

2. **Pairwise evaluation:** Implement pairwise LLM-as-judge evaluation where GPT-4o is shown two responses (A and B) and must choose which is better. Use this to compare two different RAG configurations across 50 test questions.

3. **Eval CI/CD:** Integrate the regression testing framework into a GitHub Actions workflow. Run eval on every PR that changes prompts or retrieval config. Block merges if pass_rate drops by >3%.